# Test The CUDA Warehouse SA Solver

This notebook lets you inspect the solver step by step on one of the four public test cases.

Case selector mapping:
- `1 -> PublicTestCases/Case0`
- `2 -> PublicTestCases/Case1`
- `3 -> PublicTestCases/Case2`
- `4 -> PublicTestCases/Case3`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from IPython.display import clear_output, display
import torch

from warehouse_sa_solver import (
    AnnealingState,
    SolverParams,
    evaluate_state_gpu,
    initialize_state,
    load_case,
    preprocess_case,
    run_simulated_annealing_gpu,
)

assert torch.cuda.is_available(), 'CUDA is required for this notebook.'
device = torch.device('cuda')
print('Using device:', torch.cuda.get_device_name(0))


In [ ]:
# Choose 1, 2, 3, or 4.
public_case_index = 3

case_dir = Path('PublicTestCases') / f'Case{public_case_index - 1}'
print('Selected:', case_dir)


In [ ]:
case = load_case(case_dir)
pre = preprocess_case(case, device)

print('warehouse vertices   :', len(case.warehouse_polygon))
print('warehouse boxes      :', pre.warehouse_boxes.shape[0])
print('bay types            :', len(case.bay_types))
print('obstacles            :', len(case.obstacles))
print('ceiling intervals    :', pre.ceiling_boxes.shape[0])
print('warehouse area       :', pre.warehouse_area)
print('slot cap for solver  :', pre.n_slots)
print('objective scale      :', pre.objective_scale)
print('empty state objective:', pre.empty_state_objective)


In [ ]:
def ensure_preprocessed_case():
    global case, pre
    if 'case_dir' not in globals():
        raise RuntimeError('Run the case selection cell first so case_dir is defined.')
    if 'case' not in globals() or 'pre' not in globals():
        case = load_case(case_dir)
        pre = preprocess_case(case, device)
    return case, pre


def tensor_to_xy_list(points):
    return [tuple(row) for row in points.detach().cpu().tolist()]


def build_world_polygons(state, pre):
    chain = 0
    type_id = state.type_id[chain, :, 0]
    theta_idx = state.theta_idx[chain, :, 0]
    centers = torch.stack([state.x[chain], state.y[chain]], dim=-1)
    active = state.active[chain, :, 0]

    bay_local = pre.bay_polygons_local.index_select(0, type_id)
    access_local = pre.access_polygons_local.index_select(0, type_id)
    rot = pre.rot.index_select(0, theta_idx)

    bay_world = torch.matmul(bay_local, rot.transpose(-1, -2)) + centers.unsqueeze(1)
    access_world = torch.matmul(access_local, rot.transpose(-1, -2)) + centers.unsqueeze(1)
    return active, bay_world, access_world


def theta_idx_to_degrees(theta_idx, pre):
    return float(theta_idx) * (360.0 / pre.rot.shape[0])


def summarize_state_delta(previous_state, current_state):
    if previous_state is None:
        return {'changed_from_previous': False}
    pos_delta = torch.maximum(
        torch.abs(current_state.x - previous_state.x),
        torch.abs(current_state.y - previous_state.y),
    )
    return {
        'changed_from_previous': bool(
            torch.any(current_state.active != previous_state.active).item()
            or torch.any(current_state.type_id != previous_state.type_id).item()
            or torch.any(current_state.theta_idx != previous_state.theta_idx).item()
            or torch.any(pos_delta > 1e-6).item()
        ),
        'max_position_shift': float(pos_delta.max().item()),
        'active_changed': bool(torch.any(current_state.active != previous_state.active).item()),
        'type_changed': bool(torch.any(current_state.type_id != previous_state.type_id).item()),
        'theta_changed': bool(torch.any(current_state.theta_idx != previous_state.theta_idx).item()),
    }


def plot_state(state, case, pre, title='State', max_slots_to_draw=None):
    active, bay_world, access_world = build_world_polygons(state, pre)
    fig, ax = plt.subplots(figsize=(9, 9))

    warehouse = case.warehouse_polygon
    ax.add_patch(Polygon(warehouse, closed=True, fill=False, edgecolor='black', linewidth=2.5))

    for obstacle in pre.obstacle_boxes.detach().cpu().tolist():
        xs = [p[0] for p in obstacle]
        ys = [p[1] for p in obstacle]
        ax.add_patch(Rectangle((min(xs), min(ys)), max(xs) - min(xs), max(ys) - min(ys), color='dimgray', alpha=0.5))

    indices = [i for i, is_active in enumerate(active.detach().cpu().tolist()) if is_active]
    if max_slots_to_draw is not None:
        indices = indices[:max_slots_to_draw]

    for idx in indices:
        bay = tensor_to_xy_list(bay_world[idx])
        access = tensor_to_xy_list(access_world[idx])
        ax.add_patch(Polygon(access, closed=True, facecolor='tab:orange', edgecolor='tab:orange', alpha=0.16))
        ax.add_patch(Polygon(bay, closed=True, facecolor='tab:blue', edgecolor='tab:blue', alpha=0.45))
        cx = state.x[0, idx].item()
        cy = state.y[0, idx].item()
        ax.text(cx, cy, str(idx), fontsize=8, ha='center', va='center')

    min_x, max_x, min_y, max_y = pre.warehouse_bbox.detach().cpu().tolist()
    pad_x = max(1.0, 0.05 * (max_x - min_x))
    pad_y = max(1.0, 0.05 * (max_y - min_y))
    ax.set_xlim(min_x - pad_x, max_x + pad_x)
    ax.set_ylim(min_y - pad_y, max_y + pad_y)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.show()


def summarize_eval(name, evaluation):
    print(name)
    print('  score          :', float(evaluation.score[0].item()))
    print('  objective_raw  :', float(evaluation.objective_raw[0].item()))
    print('  objective_norm :', float(evaluation.objective_norm[0].item()))
    print('  area_used      :', float(evaluation.area_used[0].item()))
    print('  access_area    :', float(evaluation.access_area[0].item()))
    print('  capacity       :', float(evaluation.total_capacity[0].item()))
    print('  cost           :', float(evaluation.total_cost[0].item()))
    print('  pair_overlap   :', float(evaluation.pair_overlap[0].item()))
    print('  obstacle_ovlp  :', float(evaluation.obstacle_overlap[0].item()))
    print('  exterior_ovlp  :', float(evaluation.exterior_overlap[0].item()))
    print('  ceiling_ovlp   :', float(evaluation.ceiling_overlap[0].item()))
    print('  feasible       :', bool(evaluation.feasible[0].item()))


def plot_snapshot(snapshot, case, pre, max_slots_to_draw=40):
    title = (
        f'Best-so-far config at batch step {snapshot.step_index + 1} '
        f'(T={snapshot.temperature:.4f}, accept={snapshot.acceptance_rate:.3f})'
    )
    summarize_eval(title, snapshot.best_evaluation)
    plot_state(snapshot.best_state, case, pre, title=title, max_slots_to_draw=max_slots_to_draw)

def make_live_snapshot_callback(case, pre, max_slots_to_draw=40, clear_previous=False):
    previous_best_state = None
    def callback(snapshot):
        nonlocal previous_best_state
        if clear_previous:
            clear_output(wait=True)
        print(
            f'Live best-so-far snapshot at batch step {snapshot.step_index + 1} '
            f'(T={snapshot.temperature:.4f}, accept={snapshot.acceptance_rate:.3f})'
        )
        display(summarize_state_delta(previous_best_state, snapshot.best_state))
        display({
            'score': float(snapshot.best_evaluation.score[0].item()),
            'objective_raw': float(snapshot.best_evaluation.objective_raw[0].item()),
            'pair_overlap': float(snapshot.best_evaluation.pair_overlap[0].item()),
            'obstacle_overlap': float(snapshot.best_evaluation.obstacle_overlap[0].item()),
            'exterior_overlap': float(snapshot.best_evaluation.exterior_overlap[0].item()),
            'ceiling_overlap': float(snapshot.best_evaluation.ceiling_overlap[0].item()),
            'feasible': bool(snapshot.best_evaluation.feasible[0].item()),
        })
        plot_state(
            snapshot.best_state,
            case,
            pre,
            title=f'Best-so-far config at batch step {snapshot.step_index + 1}',
            max_slots_to_draw=max_slots_to_draw,
        )
        previous_best_state = snapshot.best_state
    return callback


In [ ]:
case, pre = ensure_preprocessed_case()
init_params = SolverParams(n_chains=1, n_steps=1, seed=7)
initial_state = initialize_state(pre, init_params)
initial_eval = evaluate_state_gpu(initial_state, pre, init_params)

summarize_eval('Initial random state', initial_eval)
plot_state(initial_state, case, pre, title='Initial random state', max_slots_to_draw=20)


In [ ]:
case, pre = ensure_preprocessed_case()
# Optional: build your own 1-chain state manually for inspection.
# Edit the tensors below if you want to test a custom placement.

manual_state = AnnealingState(
    active=initial_state.active.clone(),
    type_id=initial_state.type_id.clone(),
    x=initial_state.x.clone(),
    y=initial_state.y.clone(),
    theta_idx=initial_state.theta_idx.clone(),
)

manual_state.active[:] = False
slots_to_keep = min(4, pre.n_slots)
manual_state.active[0, :slots_to_keep, 0] = True

manual_eval = evaluate_state_gpu(manual_state, pre, init_params)
summarize_eval('Manual state', manual_eval)
plot_state(manual_state, case, pre, title='Manual state', max_slots_to_draw=20)


In [ ]:
case, pre = ensure_preprocessed_case()
# Run a smaller annealing experiment first, then scale it up.
snapshots = []
live_snapshot_callback = make_live_snapshot_callback(case, pre, max_slots_to_draw=40, clear_previous=False)

def on_snapshot(snapshot):
    snapshots.append(snapshot)
    live_snapshot_callback(snapshot)

params = SolverParams(
    n_chains=1,
    n_steps=40,
    seed=11,
    temperature_samples=6,
    translate_step_fraction=0.03,
    toggle_probability=0.05,
    show_progress=True,
    snapshot_every_steps=1,
    penalty_weight=1e-5,
)

result = run_simulated_annealing_gpu(pre, params, snapshot_callback=on_snapshot)
summarize_eval('Best state found', result.best_evaluation)
print('mean acceptance rate:', float(result.acceptance_rates.mean().item()))
print('first temperature    :', float(result.temperatures[0].item()))
print('last temperature     :', float(result.temperatures[-1].item()))
print('snapshot count       :', len(snapshots))
print('note                 : snapshots store the best-so-far state, not every accepted state')
best_state_changed = [
    summarize_state_delta(snapshots[idx - 1].best_state if idx else None, snapshot.best_state)
    for idx, snapshot in enumerate(snapshots)
]
print('distinct best snapshots:', sum(item['changed_from_previous'] for item in best_state_changed[1:]) + (1 if best_state_changed else 0))


In [ ]:
case, pre = ensure_preprocessed_case()
# Re-plot all stored snapshots after the run if you want a static record.
for snapshot in snapshots:
    plot_snapshot(snapshot, case, pre, max_slots_to_draw=40)

plot_state(result.best_state, case, pre, title='Best final state', max_slots_to_draw=40)


In [ ]:
case, pre = ensure_preprocessed_case()
plt.figure(figsize=(8, 3.5))
plt.plot(result.acceptance_rates.detach().cpu().numpy(), marker='o', linewidth=1)
plt.title('Acceptance Rate Per Annealing Step')
plt.xlabel('Step')
plt.ylabel('Acceptance rate')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 3.5))
plt.plot(result.temperatures.detach().cpu().numpy(), marker='o', linewidth=1)
plt.title('Temperature Schedule')
plt.xlabel('Step')
plt.ylabel('Temperature')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
case, pre = ensure_preprocessed_case()
# Inspect the active slots in the best state.
best = result.best_state
active_mask = best.active[0, :, 0].detach().cpu()
active_indices = [idx for idx, flag in enumerate(active_mask.tolist()) if flag]
print('Active slot count:', len(active_indices))

for idx in active_indices[:20]:
    print(
        {
            'slot': idx,
            'type_id': int(best.type_id[0, idx, 0].item()),
            'x': float(best.x[0, idx].item()),
            'y': float(best.y[0, idx].item()),
            'theta_deg': theta_idx_to_degrees(best.theta_idx[0, idx, 0].item(), pre),
        }
    )
